# Treemün 2.0.0: single-core solver benchmark for the 105-stand bi-objective model

This notebook compares CPLEX, HiGHS, and CBC on exactly the same
weighted bi-objective model:

\[
\max\; 0.5\,\widetilde{NPV} + 0.5\,\widetilde{C},
\]

where both objectives use a single common payoff-range
normalization. The landscape contains all 105 stands. No spatial
adjacency or green-up constraints are included. Every solve requests
one solver thread.

The payoff anchors are computed once with a reference solver and are
excluded from the timed solver comparison. This ensures that every
compared solver receives identical objective coefficients.


## Scientific unit convention

The optimization coefficients and reported solutions use:

- dry aboveground biomass: **Mg**;
- biomass yield functions: **Mg ha\(^{-1}\)**;
- standing carbon: **Mg C**;
- carbon stock-time objective: **Mg C yr**;
- economic values: constant 2025 USD.

Treemün still exposes historical keys such as `ending_dry_wood_t` and
`carbon_stock_time_tC_year`. They are retained as machine-level API
identifiers, but every manuscript-facing table and export in this
notebook is renamed using Mg-based notation.


## 1. Imports and paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import pyomo.environ as pyo
import treemun_sim as tm

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

WORKING_DIRECTORY = Path.cwd().resolve()


def first_existing(candidates, label):
    for candidate in candidates:
        candidate = Path(candidate).expanduser().resolve()
        if candidate.exists():
            return candidate
    checked = "\n".join(f"  - {item}" for item in candidates)
    raise FileNotFoundError(
        f"Could not locate {label}. Checked:\n{checked}"
    )


PACKAGE_DIRECTORY = Path(tm.__file__).resolve().parent
PROJECT_ROOT = PACKAGE_DIRECTORY.parent
STANDS_FILE = first_existing(
    [
        WORKING_DIRECTORY / "forest_stands.csv",
        WORKING_DIRECTORY / "examples" / "forest_stands.csv",
        WORKING_DIRECTORY.parent / "examples" / "forest_stands.csv",
        PROJECT_ROOT / "examples" / "forest_stands.csv",
        Path("/mnt/data/benchmark_validation/forest_stands.csv"),
    ],
    "105-stand input file",
)
OUTPUT_DIRECTORY = (
    WORKING_DIRECTORY / "outputs_solver_benchmark_105_stands"
)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Treemün version:", tm.__version__)
print("Treemün location:", Path(tm.__file__).resolve())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Visible logical CPUs:", os.cpu_count())
print("Stand file:", STANDS_FILE)
print("Output directory:", OUTPUT_DIRECTORY)

In [ ]:

UNIT_MG = "Mg"
UNIT_MG_HA = r"Mg ha$^{-1}$"
UNIT_MG_C = "Mg C"
UNIT_MG_C_YR = "Mg C yr"

PAYOFF_PUBLICATION_RENAMES = {
    "economic_value": "NPV_constant_2025_USD",
    "carbon_stock_time_tC_year": "carbon_stock_time_Mg_C_yr",
    "ending_dry_wood_t": "ending_dry_wood_Mg",
}

print(
    "Reporting convention:",
    UNIT_MG,
    "for stand biomass and",
    UNIT_MG_C_YR,
    "for carbon stock-time.",
)


## 2. Reproducible benchmark configuration

In [ ]:
HORIZON = 30
DISCOUNT_RATE = 0.08
RELATIVE_GAP = 0.00

NPV_WEIGHT = 0.50
CARBON_WEIGHT = 0.50

# Fairness requirement: every solver receives exactly one thread.
SOLVER_THREADS = 1
SOLVER_TIME_LIMIT_SECONDS = None

# Repeated solves reduce the effect of timing noise. Set to 1 for a
# quick demonstration or increase for a formal computational study.
NUMBER_OF_REPETITIONS = 3
RUN_UNTIMED_WARMUP = True

FALLBACK_THINNING_FRACTION = 0.30
MINIMUM_CURVE_RESIDUAL_FRACTION = 0.10

SOLVER_INTERFACE_CANDIDATES = {
    "CPLEX": ("cplex",),
    "HiGHS": ("appsi_highs", "highs"),
    "CBC": ("cbc",),
}

CHILE_2025_ECONOMIC_PARAMETERS = {
    "revenue_per_dry_t": {
        tm.PINUS: 44.07,
        tm.EUCALYPTUS: 43.66,
    },
    "variable_cost_per_dry_t": {
        tm.PINUS: 0.0,
        tm.EUCALYPTUS: 0.0,
    },
    "transport_cost_per_dry_t": {
        tm.PINUS: 15.76,
        tm.EUCALYPTUS: 10.59,
    },
    "thinning_cost_per_dry_t": {
        tm.PINUS: 11.03,
        tm.EUCALYPTUS: 0.0,
    },
    "final_harvest_cost_per_dry_t": {
        tm.PINUS: 12.61,
        tm.EUCALYPTUS: 8.47,
    },
    "thinning_cost_per_ha": 0.0,
    "final_harvest_cost_per_ha": 0.0,
    "replanting_cost_per_ha": {
        tm.PINUS: 684.0,
        tm.EUCALYPTUS: 716.0,
    },
    "annual_management_cost_per_ha": 15.0,
    "replanting_delay_periods": 0,
    "terminal_value_per_dry_t": {
        tm.PINUS: 7.85,
        tm.EUCALYPTUS: 12.30,
    },
}

print("NPV weight:", NPV_WEIGHT)
print("Carbon weight:", CARBON_WEIGHT)
print("Relative MIP gap:", RELATIVE_GAP)
print("Threads per solver:", SOLVER_THREADS)
print("Measured repetitions:", NUMBER_OF_REPETITIONS)

## 3. Detect the available solver interfaces

In [ ]:
def interface_is_available(interface_name: str) -> bool:
    try:
        solver = pyo.SolverFactory(interface_name)
        return bool(
            solver is not None
            and solver.available(exception_flag=False)
        )
    except Exception:
        return False


available_solver_interfaces = {}
solver_detection_records = []

for solver_label, candidates in SOLVER_INTERFACE_CANDIDATES.items():
    selected_interface = None
    for candidate in candidates:
        if interface_is_available(candidate):
            selected_interface = candidate
            break
    available_solver_interfaces[solver_label] = selected_interface
    solver_detection_records.append(
        {
            "solver": solver_label,
            "selected_pyomo_interface": selected_interface,
            "available": selected_interface is not None,
        }
    )

solver_availability = pd.DataFrame(solver_detection_records)
display(solver_availability)

AVAILABLE_SOLVERS = {
    label: interface
    for label, interface in available_solver_interfaces.items()
    if interface is not None
}
if not AVAILABLE_SOLVERS:
    raise RuntimeError(
        "None of CPLEX, HiGHS, or CBC is available in this environment."
    )

REFERENCE_SOLVER_LABEL = (
    "CPLEX" if "CPLEX" in AVAILABLE_SOLVERS else next(iter(AVAILABLE_SOLVERS))
)
REFERENCE_SOLVER_INTERFACE = AVAILABLE_SOLVERS[REFERENCE_SOLVER_LABEL]

print("Available benchmark solvers:", AVAILABLE_SOLVERS)
print(
    "Reference solver for common payoff normalization:",
    REFERENCE_SOLVER_LABEL,
    f"({REFERENCE_SOLVER_INTERFACE})",
)

## 4. Simulate all 105 stands once

In [ ]:
stands = tm.load_stand_table(STANDS_FILE)

simulation_started = time.perf_counter()
(
    forest,
    policy_summary,
    ending_stock,
    harvested_by_period,
    carbon_by_period,
) = tm.simulate_forest(
    stands_file=STANDS_FILE,
    horizon=HORIZON,
    include_carbon=True,
    return_carbon_for_optimization=True,
    fallback_thinning_fraction=FALLBACK_THINNING_FRACTION,
    minimum_curve_residual_fraction=(
        MINIMUM_CURVE_RESIDUAL_FRACTION
    ),
)
simulation_time_seconds = time.perf_counter() - simulation_started

print("Input stands:", stands["stand_id"].nunique())
print("Stand-policy alternatives:", len(forest))
print("Annual trajectory rows:", sum(len(frame) for frame in forest))
print(f"Simulation time: {simulation_time_seconds:.3f} s")

assert stands["stand_id"].nunique() == 105
assert len({frame.iloc[0]["stand_id"] for frame in forest}) == 105

## 5. Build the common model specification

In [ ]:
MODEL_OPTIONS = {
    "policy_summary": policy_summary,
    "ending_dry_wood_by_policy": ending_stock,
    "harvested_dry_wood_by_period": harvested_by_period,
    "horizon": HORIZON,
    "forest": forest,
    "carbon_stock_time_by_period": carbon_by_period,
    "economic_mode": "detailed_cash_flow",
    "economic_parameters": CHILE_2025_ECONOMIC_PARAMETERS,
    "discount_rate": DISCOUNT_RATE,
    "minimum_ending_dry_wood_t": 0.0,
    "minimum_ending_dry_wood_fraction": None,
    "even_flow_mode": "none",
}

COMMON_SOLVER_OPTIONS = {
    "relative_gap": RELATIVE_GAP,
    "threads": SOLVER_THREADS,
    "time_limit_seconds": SOLVER_TIME_LIMIT_SECONDS,
    "tee": False,
}

## 6. Compute a single common payoff-range normalization

These two anchor solves are preparatory and are not included in the
reported CPLEX–HiGHS–CBC timing comparison. Their only purpose is to
define common values for \(NPV_{ref}\), \(C_{ref}\), and the two
payoff ranges.

In [ ]:
payoff_started = time.perf_counter()
common_payoff_table = tm.build_biobjective_payoff_table(
    solver_name=REFERENCE_SOLVER_INTERFACE,
    solver_options=COMMON_SOLVER_OPTIONS,
    model_options=MODEL_OPTIONS,
)
payoff_wall_time_seconds = time.perf_counter() - payoff_started
common_normalization = tm.biobjective_normalization_from_payoff(
    common_payoff_table
)

common_payoff_table_publication = (
    common_payoff_table[
        [
            "optimized_for",
            "economic_value",
            "carbon_stock_time_tC_year",
            "solve_time_seconds",
            "total_model_time_seconds",
        ]
    ]
    .rename(columns=PAYOFF_PUBLICATION_RENAMES)
)
display(common_payoff_table_publication)
common_payoff_table_publication.to_csv(
    OUTPUT_DIRECTORY / "biobjective_payoff_table_Mg_units.csv",
    index=False,
)
print("Common normalization:")
for key, value in common_normalization.items():
    print(f"  {key}: {value:,.6f}")
print(
    "Preparatory payoff wall time (excluded from benchmark):",
    f"{payoff_wall_time_seconds:.3f} s",
)

## 7. Timed weighted-model solves

In [ ]:
def build_identical_weighted_model():
    return tm.build_forest_management_model(
        **MODEL_OPTIONS,
        objective="weighted",
        npv_weight=NPV_WEIGHT,
        carbon_weight=CARBON_WEIGHT,
        **common_normalization,
    )


def selected_policy_key_set(solution) -> frozenset[tuple[str, str]]:
    selected = solution["selected_policies"]
    return frozenset(
        zip(
            selected["stand_id"].astype(str),
            selected["policy"].astype(str),
        )
    )


def solve_one_replication(
    solver_label: str,
    solver_interface: str,
    replication: int,
    measured: bool,
):
    total_started = time.perf_counter()
    build_started = time.perf_counter()
    model = build_identical_weighted_model()
    measured_build_wall_seconds = time.perf_counter() - build_started

    solve_started = time.perf_counter()
    results = tm.solve_model(
        model,
        solver_interface,
        **COMMON_SOLVER_OPTIONS,
    )
    measured_solve_wall_seconds = time.perf_counter() - solve_started
    measured_total_wall_seconds = time.perf_counter() - total_started
    solution = tm.extract_solution(model, results)

    record = {
        "solver": solver_label,
        "pyomo_interface": solver_interface,
        "replication": replication,
        "measured": measured,
        "status": str(results.solver.status),
        "termination_condition": str(
            results.solver.termination_condition
        ),
        "threads_requested": SOLVER_THREADS,
        "relative_gap_requested": RELATIVE_GAP,
        "npv_weight": NPV_WEIGHT,
        "carbon_weight": CARBON_WEIGHT,
        "economic_value": solution["economic_value"],
        "carbon_stock_time_Mg_C_yr": solution[
            "carbon_stock_time_tC_year"
        ],
        "normalized_npv_value": solution[
            "normalized_npv_value"
        ],
        "normalized_carbon_value": solution[
            "normalized_carbon_value"
        ],
        "weighted_normalized_objective": (
            NPV_WEIGHT * solution["normalized_npv_value"]
            + CARBON_WEIGHT
            * solution["normalized_carbon_value"]
        ),
        "ending_dry_wood_Mg": solution["ending_dry_wood_t"],
        "reported_model_build_time_seconds": solution[
            "model_build_time_seconds"
        ],
        "reported_solve_time_seconds": solution[
            "solve_time_seconds"
        ],
        "reported_total_model_time_seconds": solution[
            "total_model_time_seconds"
        ],
        "measured_build_wall_seconds": (
            measured_build_wall_seconds
        ),
        "measured_solve_wall_seconds": (
            measured_solve_wall_seconds
        ),
        "measured_total_wall_seconds": (
            measured_total_wall_seconds
        ),
        "number_of_selected_policies": len(
            solution["selected_policies"]
        ),
    }
    return record, selected_policy_key_set(solution)


if RUN_UNTIMED_WARMUP:
    for solver_label, solver_interface in AVAILABLE_SOLVERS.items():
        print(f"Warm-up: {solver_label} ({solver_interface})")
        solve_one_replication(
            solver_label,
            solver_interface,
            replication=0,
            measured=False,
        )

benchmark_records = []
selected_policy_sets = {}

for solver_label, solver_interface in AVAILABLE_SOLVERS.items():
    for replication in range(1, NUMBER_OF_REPETITIONS + 1):
        print(
            f"Measured solve {replication}/{NUMBER_OF_REPETITIONS}: "
            f"{solver_label} ({solver_interface})"
        )
        record, policy_set = solve_one_replication(
            solver_label,
            solver_interface,
            replication=replication,
            measured=True,
        )
        benchmark_records.append(record)
        selected_policy_sets[(solver_label, replication)] = policy_set

solver_runs = pd.DataFrame(benchmark_records)
display(solver_runs)
solver_runs.to_csv(
    OUTPUT_DIRECTORY / "solver_benchmark_individual_runs.csv",
    index=False,
)

## 8. Timing and solution-consistency summary

In [ ]:
reference_policy_set = selected_policy_sets[
    (REFERENCE_SOLVER_LABEL, 1)
]
reference_objective = float(
    solver_runs.loc[
        (solver_runs["solver"] == REFERENCE_SOLVER_LABEL)
        & (solver_runs["replication"] == 1),
        "weighted_normalized_objective",
    ].iloc[0]
)

policy_agreement_records = []
for (solver_label, replication), policy_set in selected_policy_sets.items():
    intersection = len(reference_policy_set & policy_set)
    union = len(reference_policy_set | policy_set)
    policy_agreement_records.append(
        {
            "solver": solver_label,
            "replication": replication,
            "policy_agreement_with_reference": (
                intersection / len(reference_policy_set)
            ),
            "policy_jaccard_with_reference": (
                intersection / union if union else 1.0
            ),
            "policy_hamming_distance": (
                len(reference_policy_set - policy_set)
                + len(policy_set - reference_policy_set)
            ),
        }
    )

policy_agreement = pd.DataFrame(policy_agreement_records)
solver_runs = solver_runs.merge(
    policy_agreement,
    on=["solver", "replication"],
    validate="one_to_one",
)
solver_runs["weighted_objective_difference_from_reference"] = (
    solver_runs["weighted_normalized_objective"]
    - reference_objective
)

solver_summary = (
    solver_runs.groupby(
        ["solver", "pyomo_interface"], as_index=False
    )
    .agg(
        repetitions=("replication", "size"),
        successful_runs=(
            "termination_condition",
            lambda series: int(
                series.str.lower().isin(
                    ["optimal", "maxTimeLimit".lower()]
                ).sum()
            ),
        ),
        mean_solve_time_seconds=(
            "measured_solve_wall_seconds",
            "mean",
        ),
        median_solve_time_seconds=(
            "measured_solve_wall_seconds",
            "median",
        ),
        std_solve_time_seconds=(
            "measured_solve_wall_seconds",
            "std",
        ),
        minimum_solve_time_seconds=(
            "measured_solve_wall_seconds",
            "min",
        ),
        maximum_solve_time_seconds=(
            "measured_solve_wall_seconds",
            "max",
        ),
        mean_total_time_seconds=(
            "measured_total_wall_seconds",
            "mean",
        ),
        median_total_time_seconds=(
            "measured_total_wall_seconds",
            "median",
        ),
        economic_value=("economic_value", "median"),
        carbon_stock_time_Mg_C_yr=(
            "carbon_stock_time_Mg_C_yr",
            "median",
        ),
        weighted_normalized_objective=(
            "weighted_normalized_objective",
            "median",
        ),
        policy_agreement_with_reference=(
            "policy_agreement_with_reference",
            "mean",
        ),
        policy_hamming_distance=(
            "policy_hamming_distance",
            "max",
        ),
    )
)

solver_summary["relative_solve_time_to_fastest"] = (
    solver_summary["median_solve_time_seconds"]
    / solver_summary["median_solve_time_seconds"].min()
)

display(solver_summary.sort_values("median_solve_time_seconds"))
solver_runs.to_csv(
    OUTPUT_DIRECTORY / "solver_benchmark_individual_runs.csv",
    index=False,
)
solver_summary.to_csv(
    OUTPUT_DIRECTORY / "solver_benchmark_summary.csv",
    index=False,
)

## 9. Solver-comparison figures


The figures in this section compare solver performance only. This
notebook does **not** generate an NPV--carbon Pareto frontier: it solves
one common normalized model with weights \(w_N=w_C=0.5\). A Pareto-front
figure must be generated by the complete multiobjective-analysis
notebook.


In [ ]:
ordered_summary = solver_summary.sort_values(
    "median_solve_time_seconds"
).reset_index(drop=True)

fig, axis = plt.subplots(figsize=(8, 5.5))
error_values = ordered_summary["std_solve_time_seconds"].fillna(0.0)
axis.bar(
    ordered_summary["solver"],
    ordered_summary["median_solve_time_seconds"],
    yerr=error_values,
    capsize=5,
)
axis.set_ylabel("Median solve wall time (s)")
axis.set_title(
    "Single-core weighted bi-objective solve time (0.5 NPV / 0.5 carbon)"
)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        OUTPUT_DIRECTORY / f"solver_benchmark_median_solve_time.{suffix}",
        bbox_inches="tight",
    )
plt.show()

fig, axis = plt.subplots(figsize=(8, 5.5))
axis.bar(
    ordered_summary["solver"],
    ordered_summary["median_total_time_seconds"],
)
axis.set_ylabel("Median build + solve wall time (s)")
axis.set_title("Single-core total model time")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        OUTPUT_DIRECTORY / f"solver_benchmark_median_total_time.{suffix}",
        bbox_inches="tight",
    )
plt.show()

fig, axis = plt.subplots(figsize=(8, 5.5))
axis.scatter(
    solver_runs["solver"],
    solver_runs["weighted_normalized_objective"],
)
axis.axhline(reference_objective, linestyle="--", linewidth=1)
axis.set_ylabel("Weighted normalized objective")
axis.set_title("Objective-value consistency across solvers")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        OUTPUT_DIRECTORY
        / f"solver_benchmark_weighted_objective_consistency.{suffix}",
        bbox_inches="tight",
    )
plt.show()

fig, axis = plt.subplots(figsize=(8, 5.5))
axis.bar(
    ordered_summary["solver"],
    100.0
    * ordered_summary["policy_agreement_with_reference"],
)
axis.set_ylim(0.0, 100.0)
axis.set_ylabel("Policy agreement with reference (%)")
axis.set_title("Selected-policy agreement across solvers")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        OUTPUT_DIRECTORY / f"solver_benchmark_policy_agreement.{suffix}",
        bbox_inches="tight",
    )
plt.show()

## 10. Reproducibility metadata and interpretation

In [ ]:
scheduler_metadata = {
    variable: os.getenv(variable)
    for variable in (
        "SLURM_CPUS_PER_TASK",
        "SLURM_CPUS_ON_NODE",
        "PBS_NP",
        "NSLOTS",
    )
}
reproducibility_metadata = pd.DataFrame(
    [
        {
            "treemun_version": tm.__version__,
            "python_version": sys.version.split()[0],
            "platform": platform.platform(),
            "visible_logical_cpus": os.cpu_count(),
            "threads_requested_per_solver": SOLVER_THREADS,
            "relative_gap": RELATIVE_GAP,
            "time_limit_seconds": SOLVER_TIME_LIMIT_SECONDS,
            "number_of_repetitions": NUMBER_OF_REPETITIONS,
            "warmup_excluded": RUN_UNTIMED_WARMUP,
            "reference_solver": REFERENCE_SOLVER_LABEL,
            "reference_interface": REFERENCE_SOLVER_INTERFACE,
            **scheduler_metadata,
        }
    ]
)
display(reproducibility_metadata.T.rename(columns={0: "value"}))
reproducibility_metadata.to_csv(
    OUTPUT_DIRECTORY / "solver_benchmark_metadata.csv",
    index=False,
)

print("Interpretation")
print("--------------")
print(
    "1. Compare median solve times because every solver receives "
    "the same weighted model, normalization, relative gap, and one core."
)
print(
    "2. Verify objective values and selected policies before "
    "interpreting speed differences."
)
print(
    "3. The preparatory payoff-anchor time is excluded because it "
    "is common to all compared weighted solves."
)
print(
    "4. For a formal computational experiment, run this notebook "
    "on an otherwise idle node and report hardware, solver versions, "
    "license mode, and scheduler allocation."
)
print("\nOutputs written to:", OUTPUT_DIRECTORY)

In [ ]:

solver_publication_table = solver_summary[
    [
        "solver",
        "pyomo_interface",
        "median_solve_time_seconds",
        "mean_solve_time_seconds",
        "median_total_time_seconds",
        "economic_value",
        "carbon_stock_time_Mg_C_yr",
        "weighted_normalized_objective",
        "policy_agreement_with_reference",
        "policy_hamming_distance",
    ]
].rename(
    columns={
        "economic_value": "NPV_constant_2025_USD",
        "policy_agreement_with_reference": (
            "policy_agreement_with_reference_fraction"
        ),
    }
)

display(
    solver_publication_table.sort_values(
        "median_solve_time_seconds"
    )
)
solver_publication_table.to_csv(
    OUTPUT_DIRECTORY / "solver_benchmark_publication_table.csv",
    index=False,
)

print("\nPublication figures produced by this notebook:")
for filename in [
    "solver_benchmark_median_solve_time.pdf",
    "solver_benchmark_median_total_time.pdf",
    "solver_benchmark_weighted_objective_consistency.pdf",
    "solver_benchmark_policy_agreement.pdf",
]:
    print(" -", OUTPUT_DIRECTORY / filename)
